<a href="https://colab.research.google.com/github/hamagami/IS2026/blob/main/Demo13_CNN_vs_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 13 — CNN vs. ViT: Visualizing Image Features  
# デモ13 — CNNとViTの特徴表現を可視化する

**Purpose / 目的**  
同じ画像を **ResNet18 (CNN)** と **DeiT-Tiny (Vision Transformer)** に入力し、次の3つを段階的に観察します。

1. **CNN feature maps / CNNの特徴マップ** — 浅い層から深い層への変化  
2. **ViT self-attention / ViTの自己注意** — Query（CLSまたは選択したパッチ）が、どのパッチから情報を受け取るか  
3. **Grad-CAM + occlusion / 寄与度可視化と領域隠し** — 両モデルの予測に関係する領域と、画像を変更したときの予測変化

**Important / 重要:** CNNでも画像全体の文脈を利用できます。ViTが必ず「理解が深い」という意味ではありません。Grad-CAMとAttention Weightsは異なる量です。**色が似ていても同じ意味ではありません。**

**Runtime / 実行環境:** Google Colab (CPU可、GPU推奨)。初回のみ、事前学習済み重みのダウンロードにインターネット接続が必要です。モデルの学習は行いません。

## 0. Setup / 準備
**Run all cells from top to bottom / 上から順に実行してください。**  
Colab標準のPyTorchを利用し、追加ライブラリだけを導入します。

In [ ]:
%pip -q install "timm>=1.0,<2.0" "grad-cam>=1.5,<2.0" "ipywidgets>=8,<9" "scikit-image>=0.22"

In [ ]:
import io, os, warnings, contextlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from PIL import Image, ImageDraw
from skimage import data as skdata
import torch, torchvision, timm
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from IPython.display import display, clear_output
import ipywidgets as widgets

plt.rcParams.update({'figure.dpi': 115, 'font.size': 11})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(7); np.random.seed(7)
print(f'Runtime / 実行環境: {DEVICE}; torch={torch.__version__}, timm={timm.__version__}')

## 1. Load pretrained models / 事前学習済みモデルを読み込む

- **CNN:** ResNet18 / ImageNet-1k
- **ViT:** DeiT-Tiny (16×16 patches; 14×14 = 196 image patches + 1 CLS token) / ImageNet-1k

両者の学習データやモデル規模は異なるため、**分類精度の優劣を決める実験ではありません。** 軽量で授業中に動かせるモデルを選んでいます。

In [ ]:
CNN_WEIGHTS = ResNet18_Weights.DEFAULT
cnn = models.resnet18(weights=CNN_WEIGHTS).to(DEVICE).eval()
vit = timm.create_model('deit_tiny_patch16_224.fb_in1k', pretrained=True).to(DEVICE).eval()
CLASSES = CNN_WEIGHTS.meta['categories']
# Turn off fused attention to expose post-softmax attention weights for our hooks.
for block in vit.blocks:
    block.attn.fused_attn = False
print(f'ResNet18: {sum(p.numel() for p in cnn.parameters())/1e6:.1f} M parameters')
print(f'DeiT-Tiny: {sum(p.numel() for p in vit.parameters())/1e6:.1f} M parameters, {len(vit.blocks)} blocks')
print('Model loading complete / モデル読込完了')

## 2. Select an image / 画像を選ぶ

組み込みのサンプル画像を使うか、**Upload** で好きな画像を指定します。  
表示の位置をそろえるため、**同じ正方形トリミングの224×224画像**を両モデルに与えます。学習済みモデルの標準の前処理とはわずかに異なる簡易化です。

In [ ]:
def center_square(pil):
    pil = pil.convert('RGB')
    w, h = pil.size; side = min(w, h)
    left = (w-side)//2; top = (h-side)//2
    return pil.crop((left, top, left+side, top+side)).resize((224,224), Image.Resampling.BICUBIC)

SAMPLES = {
    'Cat / 猫': Image.fromarray(skdata.chelsea()),
    'Coffee / コーヒー': Image.fromarray(skdata.coffee()),
    'Astronaut / 宇宙飛行士': Image.fromarray(skdata.astronaut()),
}
CUSTOM = None
image_choice = widgets.Dropdown(options=list(SAMPLES)+['Uploaded image / アップロード画像'], description='Image:', style={'description_width':'initial'}, layout=widgets.Layout(width='330px'))
upload_btn = widgets.Button(description='Upload image / 画像をアップロード', icon='upload', button_style='info', layout=widgets.Layout(width='260px'))
image_out = widgets.Output()

def on_upload(_):
    global CUSTOM
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded: return
        name, raw = next(iter(uploaded.items()))
        CUSTOM = Image.open(io.BytesIO(raw)).convert('RGB')
        image_choice.value = 'Uploaded image / アップロード画像'
        with image_out:
            clear_output(wait=True)
            print(f'Loaded / 読み込み: {name} | {CUSTOM.size}')
            display(center_square(CUSTOM))
    except ImportError:
        with image_out:
            clear_output(); print('Upload requires Google Colab / アップロードにはColabが必要です')

def get_image():
    im = CUSTOM if image_choice.value.startswith('Uploaded') else SAMPLES[image_choice.value]
    if im is None: raise ValueError('Please upload an image first / 画像をアップロードしてください')
    return center_square(im)

upload_btn.on_click(on_upload)
display(widgets.HBox([image_choice,upload_btn]), image_out)

MEAN = [0.485,0.456,0.406]; STD = [0.229,0.224,0.225]
TO_TENSOR = transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
def prepare(im): return TO_TENSOR(im).unsqueeze(0).to(DEVICE)
def predict(model, im):
    with torch.no_grad(): return model(prepare(im)).softmax(dim=1)[0].detach().cpu().numpy()
def topk_lines(probs, k=3):
    return [(CLASSES[int(i)], float(probs[i])) for i in np.argsort(-probs)[:k]]

def plot_preds(im):
    a,b = predict(cnn,im), predict(vit,im)
    fig, ax = plt.subplots(1,3,figsize=(14,3.8), gridspec_kw={'width_ratios':[1.2,1,1]})
    ax[0].imshow(im); ax[0].set_title('Original / 入力画像'); ax[0].axis('off')
    for idx,probs,title in [(1,a,'CNN / ResNet18'),(2,b,'ViT / DeiT-Tiny')]:
        ids=np.argsort(-probs)[:5][::-1]
        ax[idx].barh(range(5), probs[ids]*100, color=('#4984c5' if idx==1 else '#9161c8'))
        ax[idx].set_yticks(range(5),[CLASSES[int(j)][:25] for j in ids]); ax[idx].set_xlim(0,100)
        ax[idx].set_xlabel('Softmax probability (%)'); ax[idx].set_title(title)
    plt.tight_layout(); plt.show()
    return a,b

preview_btn=widgets.Button(description='Show predictions / 予測を表示',button_style='primary')
preview_out=widgets.Output()
def on_preview(_):
    with preview_out:
        clear_output(wait=True); plot_preds(get_image())
preview_btn.on_click(on_preview)
display(preview_btn,preview_out)

TypeError: 'NoneType' object is not subscriptable

## 3. CNN — From local features to deeper representations / CNN：局所特徴から階層的特徴へ

ResNet18の初期・中間・後段の特徴マップを表示します。各層で**活性値の空間的ばらつきが大きいチャネル**を例として選びます。これが意味的に重要なチャネルだという保証はありません。

**Question / 考えてみよう：** 浅い層と深い層で、特徴マップの解像度や模様はどう変わるでしょうか？

In [ ]:
CNN_LAYERS = {'Early / 浅い層 (layer1)':cnn.layer1[-1], 'Middle / 中間層 (layer2)':cnn.layer2[-1], 'Deep / 深い層 (layer4)':cnn.layer4[-1]}

def collect_cnn_maps(im):
    outputs={}; handles=[]
    for name,layer in CNN_LAYERS.items():
        handles.append(layer.register_forward_hook(lambda m, inp, out, n=name: outputs.__setitem__(n, out.detach().cpu())))
    try:
        with torch.no_grad(): cnn(prepare(im))
    finally:
        for h in handles: h.remove()
    return outputs

def show_cnn_maps(im):
    maps=collect_cnn_maps(im)
    fig,axes=plt.subplots(1,4,figsize=(14,3.6))
    axes[0].imshow(im); axes[0].set_title('Input / 入力'); axes[0].axis('off')
    for ax,(name,tensor) in zip(axes[1:],maps.items()):
        arr=tensor[0].numpy(); channel=int(np.argmax(arr.reshape(arr.shape[0],-1).var(axis=1)))
        ax.imshow(arr[channel],cmap='magma'); ax.axis('off')
        ax.set_title(f'{name.split(" (")[0]}\n{arr.shape[1]}×{arr.shape[2]}, channel {channel}',fontsize=10)
    plt.tight_layout();plt.show()

cnn_btn=widgets.Button(description='Visualize CNN layers / CNN層を見る',button_style='primary')
cnn_out=widgets.Output()
def on_cnn(_):
    with cnn_out:
        clear_output(wait=True); show_cnn_maps(get_image())
cnn_btn.on_click(on_cnn);display(cnn_btn,cnn_out)

## 4. ViT — Inspect one Attention Head / ViT：HeadごとのAttentionを見る

1. **Layer / 層** と **Head / ヘッド** を選びます。
2. **Query** をCLSトークンか、画像内の14×14パッチから選びます。
3. 他のパッチに対するAttention Weightを**ヒートマップ・重みの棒・結合線**で表示します。

**Attention Weightは「分類への寄与度」ではありません。** Softmax後の情報集約の係数です。CLSトークンも重みを受け取るため、**画像パッチに対する重みの合計は必ずしも1ではありません。**

In [ ]:
def get_attention(im, layer_index):
    store={}
    module=vit.blocks[layer_index].attn.attn_drop
    handle=module.register_forward_hook(lambda m,i,o: store.__setitem__('attn',o.detach().cpu()))
    try:
        with torch.no_grad(): vit(prepare(im))
    finally:
        handle.remove()
    if 'attn' not in store:
        raise RuntimeError('Attention hook did not run; check timm version and fused_attn=False')
    return store['attn'][0].numpy() # heads x 197 x 197

def overlay_attention(im, weights, alpha=.55):
    from PIL import Image as PILImage
    grid=np.asarray(weights).reshape(14,14)
    grid=(grid-grid.min())/(np.ptp(grid)+1e-12)
    rgba=cm.magma(grid)
    heat=PILImage.fromarray((rgba[:,:,:3]*255).astype('uint8')).resize((224,224),PILImage.Resampling.NEAREST)
    return PILImage.blend(im,heat,alpha)

def show_attention(im,layer,head,query_type,row,col):
    a=get_attention(im,layer)
    if head>=a.shape[0]: raise ValueError('Head index exceeds available heads')
    qi=0 if query_type=='CLS' else 1+14*row+col
    weights=a[head,qi,1:]
    fig,axes=plt.subplots(1,3,figsize=(15,4.8))
    axes[0].imshow(im); axes[0].set_title('Query and attended patches / 注目元')
    if qi>0:
        from matplotlib.patches import Rectangle
        axes[0].add_patch(Rectangle((col*16,row*16),16,16,fill=False,ec='#ffcf40',lw=3))
    else: axes[0].text(6,20,'CLS query (global summary)',color='white',fontsize=10,bbox=dict(facecolor='#41346c',alpha=.8))
    axes[0].axis('off')
    axes[1].imshow(overlay_attention(im,weights));axes[1].axis('off')
    axes[1].set_title(f'Layer {layer+1} / Head {head+1}\nAttention to image patches')
    ordered=np.argsort(-weights)[:9]
    cols=['#ad7ddc']*9
    axes[2].barh(np.arange(9),weights[ordered][::-1],color=cols)
    axes[2].set_yticks(np.arange(9),[f'patch ({int(k)//14},{int(k)%14})' for k in ordered[::-1]])
    axes[2].set_xlabel('Raw attention weight');axes[2].set_title('Top attended patches / 上位9パッチ')
    plt.tight_layout();plt.show()
    # Show relationships as line widths from one selected patch to its most attended patches.
    fig,ax=plt.subplots(figsize=(6,5.3));ax.imshow(im)
    if qi:
        start=(col*16+8,row*16+8)
        for rank,j in enumerate(ordered[:6]):
            end=(int(j)%14*16+8,int(j)//14*16+8)
            ax.plot([start[0],end[0]],[start[1],end[1]],color='#ffbd43',lw=1.2+4*float(weights[j]/(weights[ordered[0]]+1e-12)),alpha=.68)
        ax.scatter(*start,color='#1c467c',s=110,edgecolors='white',zorder=4)
        ax.set_title('Selected query → key patches / 結合線')
    else:
        ax.set_title('CLS query is not a spatial image patch')
    ax.axis('off');plt.tight_layout();plt.show()
    print('Sum to image patches / 画像パッチへの注意の合計 =',round(float(weights.sum()),4),
          '| CLS weight / CLSへの重み =',round(float(a[head,qi,0]),4))

layer_w=widgets.IntSlider(value=5,min=0,max=len(vit.blocks)-1,description='Layer (0-based):',style={'description_width':'initial'},continuous_update=False)
head_w=widgets.IntSlider(value=0,min=0,max=vit.blocks[0].attn.num_heads-1,description='Head (0-based):',style={'description_width':'initial'},continuous_update=False)
query_w=widgets.Dropdown(options=['CLS','Image patch / 画像パッチ'],description='Query:',style={'description_width':'initial'})
row_w=widgets.IntSlider(value=6,min=0,max=13,description='Patch row:',continuous_update=False)
col_w=widgets.IntSlider(value=6,min=0,max=13,description='Patch col:',continuous_update=False)
attn_btn=widgets.Button(description='Show Attention / Attention表示',button_style='primary')
attn_out=widgets.Output()
def on_attn(_):
    with attn_out:
        clear_output(wait=True); show_attention(get_image(),layer_w.value,head_w.value,query_w.value,row_w.value,col_w.value)
attn_btn.on_click(on_attn)
display(widgets.VBox([widgets.HBox([layer_w,head_w]),widgets.HBox([query_w,row_w,col_w]),attn_btn]),attn_out)

## 5. Same visual explanation method: Grad-CAM / 両モデルを同じ手法で可視化

**Grad-CAM** は、特定の分類スコアに関係する特徴マップを勾配を用いて可視化する手法です。ここでは同じImageNetクラスを指定し、ResNet18とDeiT-Tinyでそれぞれ求めます。

- CNN target layer: `resnet18.layer4[-1]`
- ViT target layer: `deit.blocks[-1].norm1`; CLSを除く196パッチを14×14に戻す

**注意 / Caution:** Grad-CAMは因果関係の厳密な証明ではありません。モデルの大きさ、事前学習方法も異なります。

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def vit_reshape(t):
    # [B, 197, C] -> [B, C, 14, 14], excluding CLS token
    b,n,c=t.shape
    assert n==197, f'Unexpected token count: {n}'
    return t[:,1:,:].reshape(b,14,14,c).permute(0,3,1,2)

def cam_map(model,layer,im,class_id,reshape=None):
    with GradCAM(model=model,target_layers=[layer],reshape_transform=reshape) as cam:
        result=cam(input_tensor=prepare(im), targets=[ClassifierOutputTarget(int(class_id))])
    return result[0]

def show_cam(im,target='CNN top-1'):
    pcnn=predict(cnn,im);pvit=predict(vit,im)
    target_class=int(np.argmax(pcnn if target=='CNN top-1' else pvit))
    a=cam_map(cnn,cnn.layer4[-1],im,target_class)
    b=cam_map(vit,vit.blocks[-1].norm1,im,target_class,vit_reshape)
    fig,axs=plt.subplots(1,3,figsize=(14,4.4))
    axs[0].imshow(im);axs[0].set_title('Input / 入力')
    for ax,heat,title in [(axs[1],a,'CNN / Grad-CAM'),(axs[2],b,'ViT / Grad-CAM')]:
        ax.imshow(im);ax.imshow(heat,cmap='jet',alpha=.45,vmin=0,vmax=1);ax.set_title(title)
    for ax in axs:ax.axis('off')
    fig.suptitle(f'Shared target class / 共通の対象: {CLASSES[target_class]}  |  CNN p={pcnn[target_class]:.3f} / ViT p={pvit[target_class]:.3f}',fontsize=11)
    plt.tight_layout();plt.show()
    print('Each CAM normalized separately / 各CAMは独立に0–1正規化されています（色の絶対比較不可）')

cam_target=widgets.Dropdown(options=['CNN top-1','ViT top-1'],description='Shared class:',style={'description_width':'initial'})
cam_btn=widgets.Button(description='Compare Grad-CAM / 比較表示',button_style='primary')
cam_out=widgets.Output()
def on_cam(_):
    with cam_out:
        clear_output(wait=True); show_cam(get_image(),cam_target.value)
cam_btn.on_click(on_cam);display(widgets.HBox([cam_target,cam_btn]),cam_out)

## 6. Occlusion experiment / 画像領域を隠して比較する

画像の「上・中央・下」または「左・中央・右」の一部を灰色で隠し、**各モデルの元のTop-1クラスの確率**がどう変化するかを観察します。  
**This is a perturbation experiment, not an explanation of model intentions. / これは入力変更実験であり、モデルの意図の証明ではありません。**

In [ ]:
def occlude(im,place,frac=.33):
    a=np.array(im).copy(); h,w=a.shape[:2]
    if place=='Left': a[:,0:int(w*frac)]=128
    elif place=='Center':
        x0=int(w*(1-frac)/2);x1=int(w*(1+frac)/2)
        y0=int(h*(1-frac)/2);y1=int(h*(1+frac)/2)
        a[y0:y1,x0:x1]=128
    elif place=='Right': a[:,int(w*(1-frac)):]=128
    elif place=='Top':a[0:int(h*frac),:]=128
    elif place=='Bottom':a[int(h*(1-frac)):,:]=128
    return Image.fromarray(a)

def show_occlusion(im, frac=.33):
    settings=['Original','Left','Center','Right','Top','Bottom']
    imgs=[im]+[occlude(im,s,frac) for s in settings[1:]]
    base_c=predict(cnn,im);base_v=predict(vit,im)
    cls_c=int(base_c.argmax());cls_v=int(base_v.argmax())
    probs_c=[predict(cnn,x)[cls_c] for x in imgs]
    probs_v=[predict(vit,x)[cls_v] for x in imgs]
    fig,axes=plt.subplots(1,6,figsize=(15,2.9))
    for ax,x,name in zip(axes,imgs,settings):ax.imshow(x);ax.set_title(name);ax.axis('off')
    plt.tight_layout();plt.show()
    xx=np.arange(len(settings)); width=.38
    fig,ax=plt.subplots(figsize=(11,4))
    ax.bar(xx-width/2,probs_c,width,label=f'CNN: {CLASSES[cls_c]}',color='#4788be')
    ax.bar(xx+width/2,probs_v,width,label=f'ViT: {CLASSES[cls_v]}',color='#9869bd')
    ax.set_xticks(xx,settings);ax.set_ylabel('P(original Top-1 class)');ax.set_ylim(0,1)
    ax.legend(loc='upper right');ax.grid(axis='y',alpha=.2);plt.tight_layout();plt.show()
    print('Different original top-1 classes can be compared within each model, NOT directly as identical targets.')

frac_w=widgets.FloatSlider(min=.15,max=.55,step=.05,value=.35,description='Occluded area:',style={'description_width':'initial'},continuous_update=False)
occ_btn=widgets.Button(description='Run occlusion / 領域隠しを実行',button_style='primary')
occ_out=widgets.Output()
def on_occ(_):
    with occ_out:
        clear_output(wait=True);show_occlusion(get_image(),frac_w.value)
occ_btn.on_click(on_occ);display(widgets.HBox([frac_w,occ_btn]),occ_out)

## 7. Practice / 演習（学生用）

1. **CNN feature hierarchy / CNNの階層性:** 浅い層・深い層の特徴マップでは解像度や模様がどう変化したか？  
   *How do spatial resolution and activation patterns change from shallow to deep CNN layers?*
2. **Heads / ヘッド:** 同じ画像・同じLayerでHeadを変更するとAttention Mapはどう変わるか？  
   *Do different heads assign different attention weights?*
3. **Selected query / Query変更:** CLSと画像内パッチでは重みの分布がどう異なるか？  
   *How does changing the query token change the attention distribution?*
4. **Grad-CAM comparison / 比較:** 同じクラスのGrad-CAMで共通点・相違点は何か？ 同じ色の意味は？  
   *What can—and cannot—be inferred from two heatmaps?*
5. **Occlusion / 領域を隠す:** 左・中央・右を隠したとき元のクラス確率はどう変化したか？  
   *How do predictions respond to hiding parts of the image?*

**Takeaway / 重要ポイント:**  
CNN: local operations are built into the architecture and combined hierarchically.  
ViT: image patches exchange information through input-dependent attention.  
**Neither mechanism guarantees better contextual understanding. / どちらが必ず文脈理解に優れるわけではありません。**

## References / 参考資料

- [torchvision ResNet18 weights](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [DeiT-Tiny pretrained model card](https://huggingface.co/timm/deit_tiny_patch16_224.fb_in1k)
- [timm preprocessing documentation](https://huggingface.co/docs/timm/main/en/reference/data)
- [pytorch-grad-cam — Vision Transformers](https://github.com/jacobgil/pytorch-grad-cam/blob/master/tutorials/vision_transformers.md)
- [ViT paper — An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929)

**Notes / 注意:** 事前学習済みモデルの重み取得にはネット接続が必要です。バージョンやColab環境の変更により一部APIの修正が必要になることがあります。